##### Using a Marker Column

In [ ]:
%%sql 
drop table if exists product;

create table product
(
    ProductID int
    , Name string 
    , Category string
    , Subcategory string
    , Brand string
    , Description string
    , Price float
    , Color string
    , operation_id string
    , operation_mode string
)
using delta

In [ ]:
df = spark.read.format("csv").option("header","true").load("Files/product/products.csv")
display(df)

In [ ]:
%%sql

select * from product

In [ ]:
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.functions import lit
import uuid

nof_records = df.count()

operation_id = str(uuid.uuid4())

source_with_id = df.withColumn("operation_id", lit(operation_id))


print(operation_id)


In [ ]:
deltaTable = DeltaTable.forPath(spark, "Tables/product")


if nof_records == 0:
    print("No records in the file")
else:
    merge_result = deltaTable.alias("target").merge(
        source_with_id.alias("source"),
        "target.productid = source.productid"
    ).whenMatchedUpdate(
        condition="source.is_modified == 1",
        set={
        "name": "source.name",
        "category": "source.category",
        "price": "source.price",
        "operation_id": "source.operation_id",
        "operation_mode": lit("U")
    }).whenNotMatchedInsert(values={
        "productid": "source.productid",
        "name": "source.name",
        "category": "source.category",
        "subcategory": "source.category",
        "brand": "source.brand",
        "description": "source.description",
        "price": "source.price",
        "color": "source.color",
        "operation_id": "source.operation_id",
        "operation_mode": lit("I")
    }).execute()
    print("File processed")

In [ ]:
if nof_records > 0:

    target_df = deltaTable.toDF()
    num_rows_inserted = target_df.filter(f"operation_id = '{operation_id}' AND operation_mode == 'I'").count()
    num_rows_updated = target_df.filter(f"operation_id = '{operation_id}' AND operation_mode == 'U'").count()

    print("Number of rows inserted:", num_rows_inserted)
    print("Number of rows updated:", num_rows_updated)
else:
    print("No records to update")